# DỰ ĐOÁN GIÁ NHÀ VN — 3 MÔ HÌNH ML CƠ BẢN
**Bài toán:** Hồi quy (Regression)
---
**3 Mô hình ML:** Linear Regression (Baseline), Decision Tree Regressor, Random Forest Regressor.

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import joblib, os
from sklearn.metrics import mean_squared_error, mean_absolute_error, r2_score

MODEL_DIR = os.path.join('..', 'models')
data = np.load(os.path.join(MODEL_DIR, 'preprocessed_data.npz'))
X_train, y_train = data['X_train'], data['y_train']
X_val, y_val = data['X_val'], data['y_val']
X_test, y_test = data['X_test'], data['y_test']
print('Loaded Data:', X_train.shape, X_val.shape, X_test.shape)

In [ ]:
def evaluate(model, X_train, y_train, X_val, y_val, X_test, y_test, name):
    model.fit(X_train, y_train)
    preds = model.predict(X_test)
    # Giải Log
    y_test_real = np.expm1(y_test)
    preds_real = np.expm1(preds)
    mae = mean_absolute_error(y_test_real, preds_real)
    rmse = np.sqrt(mean_squared_error(y_test_real, preds_real))
    r2 = r2_score(y_test_real, preds_real)
    return {'name': name, 'model': model, 'mae': mae, 'rmse': rmse, 'r2': r2, 'preds_real': preds_real}


In [ ]:
from sklearn.linear_model import Ridge
from sklearn.tree import DecisionTreeRegressor
from sklearn.ensemble import RandomForestRegressor

res_lr = evaluate(Ridge(alpha=1.0), X_train, y_train, X_val, y_val, X_test, y_test, 'Ridge Regression')
res_dt = evaluate(DecisionTreeRegressor(max_depth=12, random_state=42), X_train, y_train, X_val, y_val, X_test, y_test, 'Decision Tree')
res_rf = evaluate(RandomForestRegressor(n_estimators=100, max_depth=15, n_jobs=-1, random_state=42), X_train, y_train, X_val, y_val, X_test, y_test, 'Random Forest')

res_list = [res_lr, res_dt, res_rf]
df_comp = pd.DataFrame([{k:v for k,v in r.items() if k not in ['model', 'preds_real']} for r in res_list])
print(df_comp)

for r in res_list:
    joblib.dump(r['model'], os.path.join(MODEL_DIR, f"hp_{r['name'].lower().replace(' ','_')}.pkl"))
print('Saved models.')